# LangChain Smart Prompt Orchestrator

## Goal

Learn how modern LLM applications are built using LangChain.

This project will progress through several phases:

1. Foundations
2. Prompt Templates
3. Chains
4. Routing
5. Project Structure
6. FastAPI
7. Streamlit / Gradio
8. Docker and Deployment

---


```
## Final Architecture

User Request
      ↓
Router
      ↓
Choose Chain
      ↓
PromptTemplate
      ↓
LLM
      ↓
Response
```
---

This notebook begins with the foundations required to understand the architecture before building it.

# What is this notebook?

This notebook is not intended to build the final application.

Its purpose is to understand the fundamental concepts that appear in LangChain applications.

By the end of this notebook, I should understand:

- What an LLM is
- What a Prompt is
- What a PromptTemplate is
- What a Chain is
- What a Router is

These concepts will later be converted into Python modules and APIs.

In [1]:
# ==========================================
# CELL 3 — ENVIRONMENT CHECK
# ------------------------------------------
# Purpose:
# Verify that the notebook is running inside
# the correct Python environment.
# ==========================================

import sys

print("Python executable:")
print(sys.executable)

Python executable:
c:\Users\User\anaconda3\envs\llm-env\python.exe


# Foundation Concept 1 — What is an LLM?

LLM stands for Large Language Model.

A Large Language Model is a system trained on large amounts of text that can generate human-like responses.

Examples:

- GPT models
- Llama models
- Mistral models
- Gemma models

An LLM receives text as input and produces text as output.

Basic flow:

```text
User Input
      ↓
LLM
      ↓
Generated Response

# Foundation Concept 2 — What is a Prompt?

A prompt is the instruction given to an LLM.

Examples:

Prompt:
"Explain machine learning."

Prompt:
"Summarize this article."

Prompt:
"Generate three startup ideas."

The quality of the response often depends on the quality of the prompt.

Basic flow:

```text
Prompt
   ↓
LLM
   ↓
Response
```

Examples:

Prompt:
"What is machine learning?"

Response:
"Machine learning is a field of artificial intelligence that enables systems to learn patterns from data."

Prompt:
"Give me three startup ideas related to fitness."

Response:
"A fitness coaching app, a nutrition tracking platform, and an AI workout planner."

In LangChain, prompts become reusable objects called PromptTemplates.

# Foundation Concept 3 — What is a Chain?

A chain is a sequence of steps executed in a specific order.

Without a chain:

User
  ↓
LLM
  ↓
Response

With a chain:

User
  ↓
Prompt
  ↓
LLM
  ↓
Response

A chain allows us to organize the workflow.

Example:

User asks:
"Explain machine learning"

The chain may:

1. Build a prompt
2. Send the prompt to the LLM
3. Receive the response
4. Return the response to the user

Visual representation:

```text
Input
  ↓
Prompt
  ↓
LLM
  ↓
Output
```

In LangChain, chains connect components together.

A chain does not need to be complex.

Even:

Prompt → LLM

is already a chain.

# Foundation Concept 4 — What is a Router?

A router is a component that decides which workflow should be executed.

Instead of sending every request to the same chain, the router analyzes the request and chooses the most appropriate chain.

Example:

User:
"Explain machine learning"

Router:
→ explain_chain

User:
"Summarize this article"

Router:
→ summarize_chain

User:
"Generate startup ideas"

Router:
→ idea_chain

Visual representation:

```text
                 ┌───────────────┐
                 │ User Request  │
                 └───────┬───────┘
                         ↓
                  ┌───────────┐
                  │  Router   │
                  └─────┬─────┘
                        │
          ┌─────────────┼─────────────┐
          ↓             ↓             ↓
   explain_chain summarize_chain idea_chain
```

The router is responsible for deciding which chain should handle the request.

In this project, we will first build a simple router using Python rules.

Later, we will connect it to real LangChain chains.

In [2]:
# ==========================================
# CELL 8 — SIMPLE PYTHON ROUTER
# ------------------------------------------
# Purpose:
# Simulate how a router decides which
# chain should handle a user request.
#
# No LangChain yet.
# Just Python.
# ==========================================

def route_request(user_request):
    
    user_request = user_request.lower()

    if "explain" in user_request:
        return "explain_chain"

    elif "summarize" in user_request:
        return "summarize_chain"

    elif "idea" in user_request:
        return "idea_chain"

    else:
        return "default_chain"


request = "Explain machine learning"

selected_chain = route_request(request)

print("User Request:")
print(request)

print("\nSelected Chain:")
print(selected_chain)

User Request:
Explain machine learning

Selected Chain:
explain_chain


In [3]:
# ==========================================
# CELL 9 — MULTIPLE REQUESTS
# ------------------------------------------
# Purpose:
# Show how a router handles different
# types of requests.
# ==========================================

def route_request(user_request):

    user_request = user_request.lower()

    if "explain" in user_request:
        return "explain_chain"

    elif "summarize" in user_request:
        return "summarize_chain"

    elif "idea" in user_request:
        return "idea_chain"

    else:
        return "default_chain"


requests = [
    "Explain machine learning",
    "Summarize this article",
    "Give me startup ideas",
    "Tell me a joke"
]

for request in requests:

    selected_chain = route_request(request)

    print(f"Request: {request}")
    print(f"Chain: {selected_chain}")
    print("-" * 40)

Request: Explain machine learning
Chain: explain_chain
----------------------------------------
Request: Summarize this article
Chain: summarize_chain
----------------------------------------
Request: Give me startup ideas
Chain: idea_chain
----------------------------------------
Request: Tell me a joke
Chain: default_chain
----------------------------------------


In [4]:
# ==========================================
# CELL 10 — FAKE CHAINS
# ------------------------------------------
# Purpose:
# Create simple chains using Python
# functions.
#
# No LangChain yet.
# ==========================================

def explain_chain(topic):
    
    return f"Explanation about: {topic}"


def summarize_chain(text):
    
    return f"Summary of: {text}"


def idea_chain(topic):
    
    return f"Three startup ideas about: {topic}"


print(explain_chain("machine learning"))
print(summarize_chain("Artificial intelligence is transforming industries"))
print(idea_chain("fitness"))

Explanation about: machine learning
Summary of: Artificial intelligence is transforming industries
Three startup ideas about: fitness


In [5]:
# ==========================================
# CELL 11 — ROUTER EXECUTES CHAINS
# ------------------------------------------
# Purpose:
# The router now selects AND executes
# the appropriate chain.
# ==========================================

def explain_chain(topic):

    return f"Explanation about: {topic}"


def summarize_chain(text):

    return f"Summary of: {text}"


def idea_chain(topic):

    return f"Three startup ideas about: {topic}"


def route_request(user_request):

    request = user_request.lower()

    if "explain" in request:

        return explain_chain("machine learning")

    elif "summarize" in request:

        return summarize_chain("Artificial intelligence article")

    elif "idea" in request:

        return idea_chain("fitness")

    else:

        return "No chain available"


response = route_request(
    "Explain machine learning"
)

print(response)

Explanation about: machine learning


In [6]:
# ==========================================
# CELL 12 — DYNAMIC ROUTER
# ------------------------------------------
# Purpose:
# Use the user's request as input
# instead of hardcoded values.
# ==========================================

def explain_chain(user_request):

    return f"Explanation generated for: {user_request}"


def summarize_chain(user_request):

    return f"Summary generated for: {user_request}"


def idea_chain(user_request):

    return f"Startup ideas generated for: {user_request}"


def route_request(user_request):

    request = user_request.lower()

    if "explain" in request:

        return explain_chain(user_request)

    elif "summarize" in request:

        return summarize_chain(user_request)

    elif "idea" in request:

        return idea_chain(user_request)

    else:

        return "No chain available"


print(
    route_request(
        "Explain machine learning"
    )
)

print()

print(
    route_request(
        "Summarize this article"
    )
)

print()

print(
    route_request(
        "Give me startup ideas"
    )
)

Explanation generated for: Explain machine learning

Summary generated for: Summarize this article

Startup ideas generated for: Give me startup ideas


In [7]:
# ==========================================
# CELL 13 — ROUTER WITH CHAIN REGISTRY
# ------------------------------------------
# Purpose:
# Store chains in a dictionary and let
# the router select them dynamically.
# ==========================================

def explain_chain(user_request):

    return f"Explanation generated for: {user_request}"


def summarize_chain(user_request):

    return f"Summary generated for: {user_request}"


def idea_chain(user_request):

    return f"Startup ideas generated for: {user_request}"


CHAINS = {
    "explain": explain_chain,
    "summarize": summarize_chain,
    "idea": idea_chain
}


def route_request(user_request):

    request = user_request.lower()

    for keyword, chain in CHAINS.items():

        if keyword in request:

            return chain(user_request)

    return "No chain available"


requests = [
    "Explain machine learning",
    "Summarize this article",
    "Give me startup ideas",
    "Tell me a joke"
]

for request in requests:

    response = route_request(request)

    print(f"Request: {request}")
    print(f"Response: {response}")
    print("-" * 50)

Request: Explain machine learning
Response: Explanation generated for: Explain machine learning
--------------------------------------------------
Request: Summarize this article
Response: Summary generated for: Summarize this article
--------------------------------------------------
Request: Give me startup ideas
Response: Startup ideas generated for: Give me startup ideas
--------------------------------------------------
Request: Tell me a joke
Response: No chain available
--------------------------------------------------


In [3]:
# ==========================================
# CELL 14 — PROMPT TEMPLATE
# ------------------------------------------
# Purpose:
# Import LangChain's PromptTemplate
# ==========================================

from langchain_core.prompts import PromptTemplate

print("PromptTemplate imported successfully.")

PromptTemplate imported successfully.


In [4]:
from langchain_core.prompts import PromptTemplate

explain_prompt = PromptTemplate.from_template(
    """
    You are a helpful teacher.

    Explain the following topic simply:

    Topic: {topic}
    """
)

print(
    explain_prompt.format(
        topic="machine learning"
    )
)


    You are a helpful teacher.

    Explain the following topic simply:

    Topic: machine learning
    


In [5]:
# ==========================================
# CELL 15 — MULTIPLE PROMPT TEMPLATES
# ------------------------------------------
# Purpose:
# Create reusable prompts for different
# tasks.
# ==========================================

from langchain_core.prompts import PromptTemplate


explain_prompt = PromptTemplate.from_template(
    """
    You are a helpful teacher.

    Explain the following topic simply:

    Topic: {topic}
    """
)


summarize_prompt = PromptTemplate.from_template(
    """
    Summarize the following text:

    {text}
    """
)


idea_prompt = PromptTemplate.from_template(
    """
    Generate 3 startup ideas about:

    {topic}
    """
)


print("Explain Prompt")
print(
    explain_prompt.format(
        topic="machine learning"
    )
)

print("\n" + "=" * 50 + "\n")

print("Summarize Prompt")
print(
    summarize_prompt.format(
        text="Artificial intelligence is transforming industries."
    )
)

print("\n" + "=" * 50 + "\n")

print("Idea Prompt")
print(
    idea_prompt.format(
        topic="fitness"
    )
)

Explain Prompt

    You are a helpful teacher.

    Explain the following topic simply:

    Topic: machine learning
    


Summarize Prompt

    Summarize the following text:

    Artificial intelligence is transforming industries.
    


Idea Prompt

    Generate 3 startup ideas about:

    fitness
    


In [6]:
# ==========================================
# CELL 16 — LANGCHAIN CHAINS
# ------------------------------------------
# Purpose:
# Replace fake chain responses with
# PromptTemplate-generated prompts.
# ==========================================

from langchain_core.prompts import PromptTemplate


explain_prompt = PromptTemplate.from_template(
    """
    You are a helpful teacher.

    Explain the following topic simply:

    Topic: {topic}
    """
)


summarize_prompt = PromptTemplate.from_template(
    """
    Summarize the following text:

    {text}
    """
)


idea_prompt = PromptTemplate.from_template(
    """
    Generate 3 startup ideas about:

    {topic}
    """
)


def explain_chain(topic):

    return explain_prompt.format(
        topic=topic
    )


def summarize_chain(text):

    return summarize_prompt.format(
        text=text
    )


def idea_chain(topic):

    return idea_prompt.format(
        topic=topic
    )


print(explain_chain("machine learning"))

print("\n" + "=" * 50 + "\n")

print(summarize_chain(
    "Artificial intelligence is transforming industries."
))

print("\n" + "=" * 50 + "\n")

print(idea_chain("fitness"))


    You are a helpful teacher.

    Explain the following topic simply:

    Topic: machine learning
    



    Summarize the following text:

    Artificial intelligence is transforming industries.
    



    Generate 3 startup ideas about:

    fitness
    


In [7]:
import warnings
warnings.filterwarnings("ignore")

In [8]:
# ==========================================
# CELL 17 — LOCAL LLM SETUP
# ------------------------------------------
# Purpose:
# Load a small free Hugging Face model
# ==========================================

from transformers import pipeline

llm = pipeline(
    "text-generation",
    model="google/flan-t5-small",
    max_new_tokens=128
)

print("LLM loaded successfully.")

Loading weights: 100%|██████████| 190/190 [00:00<00:00, 3800.24it/s]
[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.
[transformers] The model 'T5ForConditionalGeneration' is not supported for text-generation. Supported models are ['PeftModelForCausalLM', 'AfmoeForCausalLM', 'ApertusForCausalLM', 'ArceeForCausalLM', 'AriaTextForCausalLM', 'BambaForCausalLM', 'BartForCausalLM', 'BertLMHeadModel', 'BertGenerationDecoder', 'BigBirdForCausalLM', 'BigBirdPegasusForCausalLM', 'BioGptForCausalLM', 'BitNetForCausalLM', 'BlenderbotForCausalLM', 'BlenderbotSmallForCausalLM', 'BloomForCausalLM', 'BltForCausalLM', 'CamembertForCausalLM', 'CodeGenForCausalLM', 'CohereForCausalLM', 'Cohere2ForCausalLM', 'Cohere2MoeForCausalLM', 'CpmAntForCausalLM', 'CTRLLMHeadMo

LLM loaded successfully.


In [11]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from langchain_core.prompts import PromptTemplate

# -----------------------------
# 1. Load model (NO pipeline)
# -----------------------------
model_name = "google/flan-t5-small"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

# -----------------------------
# 2. PromptTemplate
# -----------------------------
explain_prompt = PromptTemplate.from_template(
    """
    You are a helpful teacher.

    Explain the following topic simply:

    Topic: {topic}
    """
)

# -----------------------------
# 3. Create prompt
# -----------------------------
prompt = explain_prompt.format(topic="machine learning")

print("=== PROMPT ===")
print(prompt)

# -----------------------------
# 4. DIRECT GENERATION (NO PIPELINE)
# -----------------------------
inputs = tokenizer(prompt, return_tensors="pt")

outputs = model.generate(**inputs, max_new_tokens=128)

result = tokenizer.decode(outputs[0], skip_special_tokens=True)

print("\n=== RESPONSE ===")
print(result)

Loading weights: 100%|██████████| 190/190 [00:00<00:00, 3725.53it/s]
[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


=== PROMPT ===

    You are a helpful teacher.

    Explain the following topic simply:

    Topic: machine learning
    

=== RESPONSE ===
Machine learning is a learning tool.


In [8]:
prompt = explain_prompt.format(topic="machine learning")

response = llm(prompt)

print(response)

[transformers] Both `max_new_tokens` (=128) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[{'generated_text': '\n    You are a helpful teacher.\n\n    Explain the following topic simply:\n\n    Topic: machine learning\n    '}]


In [12]:
# ==========================================
# CELL 19 — SMART ORCHESTRATOR (FIRST VERSION)
# ------------------------------------------
# Goal:
# Build a simple AI system that mimics a
# LangChain-style architecture using:
#
# User Input
#     ↓
# Router (decides what to do)
#     ↓
# PromptTemplate (builds prompt)
#     ↓
# LLM (generates answer)
#
# Key idea:
# We are manually implementing what LangChain
# automates later.
# ==========================================

from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from langchain_core.prompts import PromptTemplate

# ==========================================
# 1. LOAD MODEL (shared across all chains)
# ==========================================

model_name = "google/flan-t5-small"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)


def run_llm(prompt_text):
    """
    Converts prompt → tokens → model output → readable text
    """
    inputs = tokenizer(prompt_text, return_tensors="pt")
    outputs = model.generate(**inputs, max_new_tokens=128)
    return tokenizer.decode(outputs[0], skip_special_tokens=True)


# ==========================================
# 2. PROMPT TEMPLATES
# ==========================================

explain_prompt = PromptTemplate.from_template(
    "Explain this topic simply: {topic}"
)

summarize_prompt = PromptTemplate.from_template(
    "Summarize this text: {text}"
)

idea_prompt = PromptTemplate.from_template(
    "Give 3 startup ideas about: {topic}"
)


# ==========================================
# 3. CHAINS (Prompt → LLM)
# ==========================================

def explain_chain(user_input):
    prompt = explain_prompt.format(topic=user_input)
    return run_llm(prompt)


def summarize_chain(user_input):
    prompt = summarize_prompt.format(text=user_input)
    return run_llm(prompt)


def idea_chain(user_input):
    prompt = idea_prompt.format(topic=user_input)
    return run_llm(prompt)


# ==========================================
# 4. ROUTER (decides which chain runs)
# ==========================================

def router(user_request):
    req = user_request.lower()

    if "explain" in req:
        return explain_chain(user_request)

    elif "summarize" in req:
        return summarize_chain(user_request)

    elif "idea" in req:
        return idea_chain(user_request)

    else:
        return "No valid chain found"


# ==========================================
# 5. TEST CASES
# ==========================================

tests = [
    "Explain machine learning",
    "Summarize AI is transforming industries",
    "Give me startup ideas about fitness"
]

for t in tests:
    print("\n====================")
    print("INPUT:", t)
    print("OUTPUT:", router(t))

Loading weights: 100%|██████████| 190/190 [00:00<00:00, 3166.58it/s]
[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.



INPUT: Explain machine learning
OUTPUT: Machine learning is a process that requires a machine to learn.

INPUT: Summarize AI is transforming industries
OUTPUT: AI is transforming industries

INPUT: Give me startup ideas about fitness
OUTPUT: a gym


- The router is like a traffic controller. It looks at what the user is asking and decides what type of task it is. For example, if the user says “explain machine learning,” it chooses the explanation path. If they say “summarize this text,” it chooses the summarization path. It doesn’t generate content — it only decides the direction.

- The PromptTemplate is like a structured question builder. It takes the user’s input and puts it into a clear format that the model can understand better. Instead of sending random text, it turns it into a well-designed instruction like “Explain this topic simply: machine learning.” So it defines how we ask the question.

- The LLM (language model) is the part that actually answers. It reads the formatted prompt and generates the response based on what it has learned from data. It doesn’t decide what to do or how to ask — it only focuses on producing the best possible answer.

In [2]:
# ==========================================
# CELL 20 — LCEL (STABLE VERSION)
# ------------------------------------------
# FIX:
# We separate:
#   1. Prompt formatting (LCEL)
#   2. Model execution (manual safe function)
# ==========================================

from langchain_core.prompts import PromptTemplate
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# ==========================================
# MODEL
# ==========================================

model_name = "google/flan-t5-base"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

# ==========================================
# SAFE LLM FUNCTION (expects STRING only)
# ==========================================

def run_llm(prompt_text: str) -> str:
    inputs = tokenizer(prompt_text, return_tensors="pt")
    outputs = model.generate(**inputs, max_new_tokens=128)
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

# ==========================================
# PROMPT TEMPLATE
# ==========================================

explain_prompt = PromptTemplate.from_template(
    """
You are a simple and clear teacher.

Write EXACTLY:
- 2 short sentences
- 1 real-world example

Topic: {topic}
"""
)

# ==========================================
# STEP 1 — format prompt FIRST
# ==========================================

prompt = explain_prompt.format(topic="machine learning")

# ==========================================
# STEP 2 — send clean string to model
# ==========================================

result = run_llm(prompt)

print("=== OUTPUT ===")
print(result)

Loading weights: 100%|██████████| 282/282 [00:00<00:00, 3280.14it/s]
[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


=== OUTPUT ===
Machine learning is a very important part of the human brain.


In [4]:
# ==========================================
# CELL 21 — LCEL ROUTER (FIXED VERSION)
# ------------------------------------------
# FIX:
# - NO nested .invoke()
# - NO hardcoded input
# - Proper LCEL flow
# ==========================================

from langchain_core.runnables import RunnableLambda

# ==========================================
# 1. ROUTER (simple rule-based)
# ==========================================

def router(input_dict):
    topic = input_dict["topic"].lower()

    return {
        "topic": input_dict["topic"]
    }

route_chain = RunnableLambda(router)

# ==========================================
# 2. SAFE LLM FUNCTION (unchanged)
# ==========================================

def run_llm(prompt_text: str) -> str:
    inputs = tokenizer(prompt_text, return_tensors="pt")
    outputs = model.generate(**inputs, max_new_tokens=128)
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

# ==========================================
# 3. PROMPT PIPELINE (explicit step)
# ==========================================

def build_prompt(x):
    return f"Explain this topic simply in 2 sentences with an example: {x['topic']}"

prompt_chain = RunnableLambda(build_prompt)

# ==========================================
# 4. FULL PIPELINE (CLEAN LCEL STYLE)
# ==========================================

full_chain = route_chain | prompt_chain | RunnableLambda(run_llm)

# ==========================================
# 5. TEST
# ==========================================

result = full_chain.invoke({"topic": "machine learning"})

print("=== OUTPUT ===")
print(result)

=== OUTPUT ===
machine learning


In [6]:
# ==========================================
# CELL 23 — LCEL ROUTER (FULL STABLE VERSION)
# ------------------------------------------
# Goal:
# - RunnableBranch routing
# - SAFE input handling for HuggingFace model
# - No type errors between LCEL and tokenizer
# ==========================================

from langchain_core.runnables import RunnableLambda, RunnableBranch
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import PromptTemplate

# ==========================================
# 1. PROMPT TEMPLATE
# ==========================================

explain_prompt = PromptTemplate.from_template(
    "Explain this topic simply in 2 sentences with an example:\n{topic}"
)

# ==========================================
# 2. SAFE LLM FUNCTION (CRITICAL FIX)
# ==========================================
# This prevents crashes when LCEL sends dicts or objects

def run_llm(prompt_text):
    # Normalize input to string (VERY IMPORTANT)
    if isinstance(prompt_text, dict):
        prompt_text = prompt_text.get("topic", str(prompt_text))

    prompt_text = str(prompt_text)

    inputs = tokenizer(prompt_text, return_tensors="pt")
    outputs = model.generate(**inputs, max_new_tokens=128)

    return tokenizer.decode(outputs[0], skip_special_tokens=True)

# ==========================================
# 3. EXPLAIN CHAIN
# ==========================================

explain_chain = (
    explain_prompt
    | RunnableLambda(run_llm)
    | StrOutputParser()
)

# ==========================================
# 4. ROUTING CONDITION
# ==========================================
# Simple rule-based classifier (we will upgrade later)

def is_explain(input_dict):
    topic = input_dict.get("topic", "").lower()

    return (
        "explain" in topic or
        "what" in topic or
        "machine" in topic
    )

# ==========================================
# 5. FALLBACK CHAIN
# ==========================================

fallback_chain = RunnableLambda(
    lambda x: "I can only explain topics for now."
)

# ==========================================
# 6. LCEL ROUTER (RunnableBranch)
# ==========================================

router_chain = RunnableBranch(
    (is_explain, explain_chain),
    fallback_chain
)

# ==========================================
# 7. TEST EXECUTION
# ==========================================

result = router_chain.invoke(
    {"topic": "machine learning"}
)

print("=== LCEL ROUTED OUTPUT ===")
print(result)

=== LCEL ROUTED OUTPUT ===
Machine learning is a field in which humans learn to use machines.


In [7]:
# ==========================================
# CELL 24 — LLM ROUTER (INTELLIGENT ROUTING)
# ==========================================
# PURPOSE:
# Build a routing system where an LLM decides
# which chain to execute.
#
# ARCHITECTURE:
# User Input
#   ↓
# Router LLM (decides: explain or other)
#   ↓
# RunnableBranch (routes execution)
#   ↓
# Final Chain Output
# ==========================================

from langchain_core.runnables import RunnableLambda, RunnableBranch
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

# ==========================================
# 1. ROUTER PROMPT (LLM DECISION MAKER)
# ==========================================
# This prompt forces the model to classify the request

router_prompt = PromptTemplate.from_template(
    """
You are a routing system.

Your job is to decide what to do with the user input.

Return ONLY ONE word:
- explain
- other

User input:
{topic}
"""
)

# ==========================================
# 2. SAFE LLM FUNCTION
# ==========================================
# Ensures compatibility between LangChain and HF model

def run_llm(prompt_text):
    # Normalize input type (VERY IMPORTANT)
    if isinstance(prompt_text, dict):
        prompt_text = prompt_text.get("topic", str(prompt_text))

    prompt_text = str(prompt_text)

    inputs = tokenizer(prompt_text, return_tensors="pt")
    outputs = model.generate(**inputs, max_new_tokens=128)

    return tokenizer.decode(outputs[0], skip_special_tokens=True)

# ==========================================
# 3. ROUTER CHAIN (LLM DECISION)
# ==========================================
# The LLM decides the route (not rules anymore)

router_chain = (
    router_prompt
    | RunnableLambda(run_llm)
    | StrOutputParser()
)

# ==========================================
# 4. EXPLAIN CHAIN
# ==========================================
# This chain is executed when route = "explain"

explain_prompt = PromptTemplate.from_template(
    "Explain this topic simply in 2 sentences with an example:\n{topic}"
)

explain_chain = (
    explain_prompt
    | RunnableLambda(run_llm)
    | StrOutputParser()
)

# ==========================================
# 5. ROUTE DECISION FUNCTION
# ==========================================
# Converts LLM output into a clean routing label

def route_selector(route_output):
    route_output = route_output.strip().lower()

    if "explain" in route_output:
        return "explain"
    else:
        return "other"

# ==========================================
# 6. FALLBACK CHAIN
# ==========================================

fallback_chain = RunnableLambda(
    lambda x: "I can only explain topics for now."
)

# ==========================================
# 7. FULL LCEL PIPELINE
# ==========================================
# Flow:
# Input → Router LLM → Branch → Chain → Output

full_chain = (
    router_chain
    | RunnableLambda(lambda route: {
        "route": route_selector(route),
        "topic": "machine learning"
    })
    | RunnableBranch(
        (lambda x: x["route"] == "explain", explain_chain),
        fallback_chain
    )
)

# ==========================================
# 8. TEST EXECUTION
# ==========================================

result = full_chain.invoke({"topic": "machine learning"})

print("=== LLM ROUTER OUTPUT ===")
print(result)

=== LLM ROUTER OUTPUT ===
I can only explain topics for now.


In [9]:
# ==========================================================
# CELL 25 — FIX LCEL + TOKENIZER CONTRACT (STABLE VERSION)
# ----------------------------------------------------------
# PURPOSE:
# Fix the mismatch between LangChain dict inputs and
# HuggingFace tokenizer string requirements.
#
# RULE:
# LCEL passes dict → we must extract "topic" safely.
# ==========================================================

from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnableLambda, RunnableBranch
from langchain_core.output_parsers import StrOutputParser

# ----------------------------------------------------------
# 1. ROUTER FUNCTION
# ----------------------------------------------------------
def route_input(inputs: dict) -> str:
    text = inputs["topic"].lower()

    if "example" in text:
        return "example"
    elif "what" in text or "explain" in text:
        return "explain"
    else:
        return "fallback"


# ----------------------------------------------------------
# 2. PROMPTS
# ----------------------------------------------------------
explain_prompt = PromptTemplate.from_template(
    "Explain this topic simply:\n{topic}"
)

example_prompt = PromptTemplate.from_template(
    "Give a simple real-world example of:\n{topic}"
)

fallback_prompt = PromptTemplate.from_template(
    "Answer briefly:\n{topic}"
)


# ----------------------------------------------------------
# 3. SAFE LLM WRAPPER (IMPORTANT FIX)
# ----------------------------------------------------------
def run_llm(inputs: dict) -> str:
    """
    FIX:
    LCEL passes dict → we extract topic safely → convert to string
    """
    if isinstance(inputs, dict):
        text = inputs.get("topic", "")
    else:
        text = str(inputs)

    inputs_tok = tokenizer(text, return_tensors="pt")
    outputs = model.generate(**inputs_tok, max_new_tokens=128)
    return tokenizer.decode(outputs[0], skip_special_tokens=True)


# ----------------------------------------------------------
# 4. CHAINS
# ----------------------------------------------------------
explain_chain = explain_prompt | RunnableLambda(run_llm) | StrOutputParser()
example_chain = example_prompt | RunnableLambda(run_llm) | StrOutputParser()
fallback_chain = fallback_prompt | RunnableLambda(run_llm) | StrOutputParser()


# ----------------------------------------------------------
# 5. ROUTER + BRANCHING
# ----------------------------------------------------------
full_chain = RunnableBranch(
    (lambda x: route_input(x) == "explain", explain_chain),
    (lambda x: route_input(x) == "example", example_chain),
    fallback_chain
)


# ----------------------------------------------------------
# 6. TEST
# ----------------------------------------------------------
result = full_chain.invoke({"topic": "machine learning example"})

print("=== LCEL ROUTED OUTPUT ===")
print(result)

=== LCEL ROUTED OUTPUT ===
a machine learning example


In [10]:
# ==========================================================
# CELL 26 — PROMPT IMPROVEMENT (LCEL QUALITY UPGRADE)
# ----------------------------------------------------------
# PURPOSE:
# Improve the quality of responses without changing the
# architecture of the LCEL router system.
#
# WHAT THIS DOES:
# - Makes explanations clearer and more structured
# - Forces concrete examples (not vague answers)
# - Improves consistency of fallback responses
# ==========================================================

from langchain_core.prompts import PromptTemplate

# ----------------------------------------------------------
# 1. EXPLAIN PROMPT (improved teaching behavior)
# ----------------------------------------------------------
# Goal:
# - Simple explanation
# - Structured thinking
# - Avoid vague definitions
explain_prompt = PromptTemplate.from_template(
    """
You are a helpful and clear teacher.

Explain the topic in a simple way for a beginner.

Rules:
- Use 2 to 3 short sentences
- Avoid repetition
- Use a real-world analogy if possible

Topic: {topic}
"""
)

# ----------------------------------------------------------
# 2. EXAMPLE PROMPT (make answers concrete)
# ----------------------------------------------------------
# Goal:
# - Force real-world grounding
# - Avoid abstract or generic answers
example_prompt = PromptTemplate.from_template(
    """
You are a teacher giving practical learning examples.

Give ONE clear real-world example of the topic.

Rules:
- Be specific (use a real situation or scenario)
- Keep it in 2–3 sentences max
- Do not explain theory, only example

Topic: {topic}
"""
)

# ----------------------------------------------------------
# 3. FALLBACK PROMPT (control weak routing cases)
# ----------------------------------------------------------
# Goal:
# - Prevent empty or vague outputs
# - Force short structured response
fallback_prompt = PromptTemplate.from_template(
    """
You are a helpful assistant.

Answer the topic in a simple and direct way.

Rules:
- 1 to 2 short sentences only
- Be clear and concrete
- Avoid vague statements

Topic: {topic}
"""
)

In [11]:
# ==========================================================
# CELL 26 — PROMPT IMPROVEMENT (LCEL QUALITY UPGRADE)
# ----------------------------------------------------------
# PURPOSE:
# Improve the quality of responses without changing the
# architecture of the LCEL router system.
#
# WHAT THIS DOES:
# - Makes explanations clearer and more structured
# - Forces concrete examples (not vague answers)
# - Improves consistency of fallback responses
# ==========================================================

from langchain_core.prompts import PromptTemplate

# ----------------------------------------------------------
# 1. EXPLAIN PROMPT (improved teaching behavior)
# ----------------------------------------------------------
# Goal:
# - Simple explanation
# - Structured thinking
# - Avoid vague definitions
explain_prompt = PromptTemplate.from_template(
    """
You are a helpful and clear teacher.

Explain the topic in a simple way for a beginner.

Rules:
- Use 2 to 3 short sentences
- Avoid repetition
- Use a real-world analogy if possible

Topic: {topic}
"""
)

# ----------------------------------------------------------
# 2. EXAMPLE PROMPT (make answers concrete)
# ----------------------------------------------------------
# Goal:
# - Force real-world grounding
# - Avoid abstract or generic answers
example_prompt = PromptTemplate.from_template(
    """
You are a teacher giving practical learning examples.

Give ONE clear real-world example of the topic.

Rules:
- Be specific (use a real situation or scenario)
- Keep it in 2–3 sentences max
- Do not explain theory, only example

Topic: {topic}
"""
)

# ----------------------------------------------------------
# 3. FALLBACK PROMPT (control weak routing cases)
# ----------------------------------------------------------
# Goal:
# - Prevent empty or vague outputs
# - Force short structured response
fallback_prompt = PromptTemplate.from_template(
    """
You are a helpful assistant.

Answer the topic in a simple and direct way.

Rules:
- 1 to 2 short sentences only
- Be clear and concrete
- Avoid vague statements

Topic: {topic}
"""
)

In [12]:
result = full_chain.invoke({"topic": "machine learning example"})
print(result)

a machine learning example


In [13]:
# ==========================================================
# CELL 27 — FIX LCEL → HF TOKENIZER INTERFACE (CRITICAL FIX)
# ----------------------------------------------------------
# PURPOSE:
# Ensure PromptTemplate output is converted into a proper
# string BEFORE being passed to HuggingFace tokenizer.
#
# WHY THIS IS NEEDED:
# LangChain passes structured inputs (dict-like),
# but HF tokenizer requires a raw string.
# ==========================================================

from langchain_core.runnables import RunnableLambda
from langchain_core.output_parsers import StrOutputParser

# ----------------------------------------------------------
# SAFE WRAPPER (THIS IS THE KEY FIX)
# ----------------------------------------------------------
def run_llm(inputs):
    """
    Converts LangChain structured input → clean string → HF model
    """

    # If LCEL sends dict, extract topic safely
    if isinstance(inputs, dict):
        topic = inputs.get("topic", "")
    else:
        topic = str(inputs)

    # Build final prompt string manually (IMPORTANT FIX)
    prompt_text = f"Explain this topic simply:\n{topic}"

    # Tokenize + generate
    inputs_tok = tokenizer(prompt_text, return_tensors="pt")
    outputs = model.generate(**inputs_tok, max_new_tokens=128)

    return tokenizer.decode(outputs[0], skip_special_tokens=True)


# ----------------------------------------------------------
# REBUILD CHAIN (KEEP SAME STRUCTURE)
# ----------------------------------------------------------
explain_chain = explain_prompt | RunnableLambda(run_llm) | StrOutputParser()


# ----------------------------------------------------------
# TEST
# ----------------------------------------------------------
result = explain_chain.invoke({"topic": "machine learning"})
print("=== OUTPUT ===")
print(result)

=== OUTPUT ===
Machine learning is a field that is often referred to as machine learning.


In [14]:
# ==========================================================
# CELL 28 — CLEAN LCEL CHAIN (NO MANUAL TOKENIZER)
# ----------------------------------------------------------
# PURPOSE:
# Replace manual HuggingFace function with a clean LCEL-compatible
# runnable function.
#
# WHY:
# - Removes ValueError issues from tokenizer
# - Aligns architecture with real LangChain LCEL design
# - Makes pipeline stable and predictable
# ==========================================================

from langchain_core.runnables import RunnableLambda
from langchain_core.output_parsers import StrOutputParser

# ----------------------------------------------------------
# SIMPLE SAFE LLM WRAPPER (LCEL COMPATIBLE)
# ----------------------------------------------------------
def llm_runnable(input_dict):
    """
    Receives LCEL dict input → extracts topic → runs HF model safely
    """

    # Extract topic safely from LCEL input
    topic = input_dict.get("topic", "")

    # Build clean prompt
    prompt_text = f"Explain this topic simply in 2-3 sentences:\n{topic}"

    # HuggingFace inference (SAFE + CONSISTENT)
    inputs = tokenizer(prompt_text, return_tensors="pt")
    outputs = model.generate(**inputs, max_new_tokens=128)

    return tokenizer.decode(outputs[0], skip_special_tokens=True)


# ----------------------------------------------------------
# FINAL LCEL EXPLAIN CHAIN
# ----------------------------------------------------------
explain_chain = explain_prompt | RunnableLambda(llm_runnable) | StrOutputParser()

print("LCEL explain_chain ready ✔")

LCEL explain_chain ready ✔


In [15]:
# ==========================================================
# CELL 28 — CLEAN LCEL CHAIN (NO MANUAL TOKENIZER)
# ----------------------------------------------------------
# PURPOSE:
# Replace manual HuggingFace function with a clean LCEL-compatible
# runnable function.
#
# WHY:
# - Removes ValueError issues from tokenizer
# - Aligns architecture with real LangChain LCEL design
# - Makes pipeline stable and predictable
# ==========================================================

from langchain_core.runnables import RunnableLambda
from langchain_core.output_parsers import StrOutputParser

# ----------------------------------------------------------
# SIMPLE SAFE LLM WRAPPER (LCEL COMPATIBLE)
# ----------------------------------------------------------
def llm_runnable(input_dict):
    """
    Receives LCEL dict input → extracts topic → runs HF model safely
    """

    # Extract topic safely from LCEL input
    topic = input_dict.get("topic", "")

    # Build clean prompt
    prompt_text = f"Explain this topic simply in 2-3 sentences:\n{topic}"

    # HuggingFace inference (SAFE + CONSISTENT)
    inputs = tokenizer(prompt_text, return_tensors="pt")
    outputs = model.generate(**inputs, max_new_tokens=128)

    return tokenizer.decode(outputs[0], skip_special_tokens=True)


# ----------------------------------------------------------
# FINAL LCEL EXPLAIN CHAIN
# ----------------------------------------------------------
explain_chain = explain_prompt | RunnableLambda(llm_runnable) | StrOutputParser()

print("LCEL explain_chain ready ✔")

LCEL explain_chain ready ✔


In [16]:
# ==========================================================
# CELL 30 — FINAL DEMO (NOTEBOOK COMPLETION)
# ----------------------------------------------------------
# PURPOSE:
# Single clean demonstration of entire system:
# input → router → chain → output
#
# THIS IS THE END OF NOTEBOOK 1
# ==========================================================

inputs = [
    {"topic": "machine learning"},
    {"topic": "machine learning example"},
    {"topic": "explain deep learning"}
]

for inp in inputs:
    print("\nINPUT:", inp["topic"])
    print("OUTPUT:", router_chain.invoke(inp))
    print("-" * 50)


INPUT: machine learning
OUTPUT: fallback
--------------------------------------------------

INPUT: machine learning example
OUTPUT: example
--------------------------------------------------

INPUT: explain deep learning
OUTPUT: explain
--------------------------------------------------
